# 🛰️ Alternative data — insider, analyst ratings, price targets, short interest

A peer of `01_universe_screener` and `02_backfill`, but for the **non-OHLCV**
signals that drive the `informed_flow` strategy. Everything here is **this
repo's own ingestion** — SEC EDGAR (Form 4 insider), Yahoo Finance (analyst
rating changes **and price targets**), FINRA (short interest) — writing to the
**shared TimescaleDB**. No dependency on the sibling `stock-trading` repo.

**This notebook is the single entry point for alt-data ingestion**, incremental
*and* historic. Set `MODE = "backfill"` to (re)load deep history; there is no
separate CLI recipe to remember.

Point-in-time by construction: every row is stamped `known_at` (SEC filing
acceptance, FINRA dissemination, rating/target announcement), so features never
see a value before it was public. Writes UPSERT on natural keys, so re-running
is safe, cheap, and idempotent — a backfill simply fills the gaps.

> **Why the coverage audit in §5 matters.** A half-ingested table does not fail
> loudly — it goes *quiet*. `short_interest` once shipped covering 3 of 100
> names, and because the freshness check only looked at the *newest* row, it read
> as perfectly fresh. Meanwhile the strategy handed those 3 names real
> cross-sectional ranks and gave the other 97 a neutral 0.5, turning the sleeve
> into a large arbitrary tilt on three mega-caps. §5 checks **breadth and depth**,
> not just recency, so that cannot happen silently again.

Flow: **parameters → ingest → freshness → per-symbol coverage & gaps → schema.**

## 1 · Parameters

In [ ]:
# MODE picks how far back to ingest. Everything else is identical — same paced
# subprocess, same idempotent UPSERT — so a backfill is just an update with a
# deeper SINCE.
#   "update"   — keep the tables current (run this daily/weekly).
#   "backfill" — (re)load deep history. Run after adding a dataset, after finding
#                a coverage hole in §5, or on a fresh database.
MODE = "backfill"

UNIVERSE = "LiquidLeaders"     # names to keep fresh (the informed_flow universe)
DATASETS = ["insider", "ratings", "price_target", "short_interest"]
SYMBOLS  = None                # None = the whole universe; or a short list to smoke-test

# We ingest from 2020 even though informed_flow TRAINS from 2022 (its config's
# data.start). The two are deliberately separate knobs: this history is free, so
# holding it means widening the train window later is a one-line config change
# rather than a re-ingestion. Deep history costs a slow first run, nothing after.
BACKFILL_START = "2020-01-01"
UPDATE_START   = "2025-01-01"  # an overlap window; UPSERT dedupes the rest

SINCE = BACKFILL_START if MODE == "backfill" else UPDATE_START

# SEC EDGAR REQUIRES a contact email in the User-Agent or it returns 403.
# Put your own name + email here (SEC uses it only to reach you if you cause load).
import os
os.environ.setdefault("TRADING_SEC_USER_AGENT", "your-name your-email@example.com")

print(f"MODE={MODE!r} -> ingesting since {SINCE}")

## 2 · Set up

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run
#   uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from datetime import date
import pandas as pd
from sqlalchemy import text

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import jobs

settings = load_settings()
engine = make_engine(settings)
symbols = SYMBOLS or settings.universe_symbols(UNIVERSE)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")
print(f"{UNIVERSE}: {len(symbols)} symbols; datasets: {DATASETS}")

## 3 · Ingest each dataset

Each dataset runs as **one paced `tsim altdata` subprocess** (mirroring the
backfill handoff). SEC and FINRA rate-limit per IP, so — like the IBKR rule —
there is deliberately **no fan-out**; per-symbol progress streams below. Insider
is the slow one (a request per Form 4 filing); ratings, price targets and short
interest are quick. Re-running only writes rows not already stored (idempotent
UPSERT), so this is safe to run daily *and* safe to re-run as a backfill.

`ratings` and `price_target` come from the **same** Yahoo response, so ingesting
both costs no extra requests — they are two tables because they are two signals:
the large majority of target revisions arrive with the *rating unchanged*, and
the rating stream scores those as exactly zero.

⏳ A full `MODE="backfill"` over 100 names takes a while (insider dominates).
Everything below is resumable — if it dies, just run it again.

In [ ]:
for ds in DATASETS:
    print(f"\n=== {ds} ===")
    jobs.run_altdata(ds, since=SINCE, universe=UNIVERSE, symbols=SYMBOLS)

## 4 · Freshness, breadth & depth

Three questions, not one. **Fresh** (is the newest row recent?), **broad** (how
many of the universe's names are present?) and **deep** (how far back does it
go?).

The old version of this check asked only the first, which is exactly how a
3-of-100-symbol `short_interest` table passed as healthy. `first_known_at` and
`coverage` are the columns that catch that — a `⚠` in either means the sleeve is
**not safe to trade on**: `informed_flow` will drop any signal covering less than
half the universe, but a sleeve that is broad enough to survive that floor yet
still patchy will quietly distort the ranking.

In [ ]:
TABLES = {"insider": "insider_trades", "ratings": "analyst_ratings",
          "price_target": "analyst_price_targets", "short_interest": "short_interest"}
MIN_COVERAGE = 0.5   # informed_flow drops a sleeve below this (strategy._MIN_COVERAGE)

uni = settings.universe_symbols(UNIVERSE)
want_start = pd.Timestamp(BACKFILL_START, tz="UTC")
rows = []
with engine.connect() as c:
    for label, tbl in TABLES.items():
        stat = c.execute(text(
            f"SELECT count(*), min(known_at), max(known_at), count(DISTINCT symbol) "
            f"FROM {tbl} WHERE symbol = ANY(:s)"), {"s": uni}).one()
        rows.append({"dataset": label, "table": tbl, "rows": stat[0],
                     "first_known_at": stat[1], "last_known_at": stat[2],
                     "symbols": stat[3]})

report = pd.DataFrame(rows)
today = pd.Timestamp(date.today(), tz="UTC")
report["coverage"] = report["symbols"] / len(uni)
report["stale_days"] = report["last_known_at"].apply(
    lambda t: (today - pd.Timestamp(t)).days if t is not None else None)
# Depth: how much later than BACKFILL_START does the history actually begin?
report["short_by_days"] = report["first_known_at"].apply(
    lambda t: max(0, (pd.Timestamp(t) - want_start).days) if t is not None else None)

def _flag(row):
    """Why this sleeve is not trustworthy — empty string when it is."""
    if not row["rows"]:
        return "⚠ EMPTY — run MODE='backfill'"
    if row["coverage"] < MIN_COVERAGE:
        return f"⚠ THIN — {row['symbols']}/{len(uni)} names; informed_flow will DROP it"
    if row["short_by_days"] and row["short_by_days"] > 90:
        return f"⚠ SHALLOW — history starts {row['short_by_days']}d after {BACKFILL_START}"
    return ""

report["verdict"] = report.apply(_flag, axis=1)
display(report[["dataset", "rows", "symbols", "coverage", "first_known_at",
                "last_known_at", "stale_days", "verdict"]]
        .style.format({"coverage": "{:.0%}"}).hide(axis="index"))

if (bad := report[report["verdict"] != ""]).empty:
    print(f"✅ all {len(report)} datasets cover ≥{MIN_COVERAGE:.0%} of {UNIVERSE} "
          f"back to {BACKFILL_START}")
else:
    print("❌ not safe to trade on:")
    for _, r in bad.iterrows():
        print(f"   {r['dataset']:15s} {r['verdict']}")

In [ ]:
import plotly.graph_objects as go
from trading_simple.plots import SERIES

fig = go.Figure(go.Bar(
    x=report["dataset"], y=report["coverage"],
    marker_color=[SERIES[5] if v < MIN_COVERAGE else SERIES[0] for v in report["coverage"]],
    text=[f"{s} names" for s in report["symbols"]], textposition="outside"))
fig.add_hline(y=MIN_COVERAGE, line={"dash": "dash", "color": SERIES[5]},
              annotation_text=f"informed_flow drops a sleeve below {MIN_COVERAGE:.0%}")
fig.update_layout(
    title=f"Alt-data breadth — share of {UNIVERSE} covered (red = will be dropped)",
    template="plotly_white", height=360,
    yaxis={"title": "universe coverage", "tickformat": ".0%", "range": [0, 1.15]})
fig.show()

## 5 · Per-symbol coverage & gaps

The table above is an average, and an average hides exactly the failure that
matters: a dataset can look 80% covered while a fifth of the universe has *no
data at all*. This section goes name by name.

**What counts as a gap depends on the dataset**, and pretending otherwise would
invent precision the data cannot support:

- `short_interest` is **scheduled** — FINRA disseminates roughly twice a month.
  So a missing settlement really is a gap, and we can check observed vs expected
  counts per symbol.
- `insider`, `ratings` and `price_target` are **irregular event streams**. A
  quiet stretch is not a gap: plenty of companies simply have no Form 4 filings
  for months. Flagging "silence" as a fault would cry wolf constantly. What *is*
  meaningful: a name with **zero events over the whole window** (never covered —
  it will ride the neutral 0.5 rank forever), and a name whose **longest silence
  is wildly out of line with its own history** (suggesting an interrupted
  ingest).

In [ ]:
# One row per (symbol, dataset): when its history starts/ends, how many events,
# and the longest silence. Pulled as one query per table, then pivoted.
SCHEDULED = {"short_interest"}          # ~24 disseminations a year
EXPECTED_PER_YEAR = 24

events = {}
with engine.connect() as c:
    for label, tbl in TABLES.items():
        events[label] = pd.read_sql(text(
            f"SELECT symbol, (known_at AT TIME ZONE 'UTC')::date AS d, count(*) AS n "
            f"FROM {tbl} WHERE symbol = ANY(:s) AND known_at >= :since "
            f"GROUP BY symbol, d ORDER BY symbol, d"),
            c, params={"s": uni, "since": want_start.to_pydatetime()},
            parse_dates=["d"])

def _per_symbol(label: str, frame: pd.DataFrame) -> pd.DataFrame:
    """First/last event, count, and longest silence for every universe name."""
    out = []
    span_years = max((today - want_start).days / 365.25, 1e-9)
    for sym in uni:
        g = frame[frame["symbol"] == sym]
        if g.empty:
            out.append({"symbol": sym, "dataset": label, "events": 0, "first": pd.NaT,
                        "last": pd.NaT, "max_gap_d": None, "verdict": "⚠ NEVER COVERED"})
            continue
        dates = pd.to_datetime(g["d"])
        # Silence includes the tail: the stretch from the last event until today.
        edges = pd.concat([pd.Series([want_start.tz_localize(None)]), dates,
                           pd.Series([today.tz_localize(None)])])
        max_gap = int(edges.diff().max().days)
        n = int(g["n"].sum())
        verdict = ""
        if label in SCHEDULED:
            expected = EXPECTED_PER_YEAR * span_years
            if n < 0.7 * expected:
                verdict = f"⚠ SPARSE — {n} of ~{expected:.0f} expected disseminations"
        elif max_gap > 400:
            verdict = f"⚠ {max_gap}d silence — check for an interrupted ingest"
        out.append({"symbol": sym, "dataset": label, "events": n,
                    "first": dates.min(), "last": dates.max(),
                    "max_gap_d": max_gap, "verdict": verdict})
    return pd.DataFrame(out)

per_symbol = pd.concat([_per_symbol(k, v) for k, v in events.items()], ignore_index=True)

problems = per_symbol[per_symbol["verdict"] != ""]
print(f"{len(uni)} symbols x {len(TABLES)} datasets = {len(per_symbol)} pairs; "
      f"{len(problems)} flagged\n")
if problems.empty:
    print("✅ every symbol has usable history in every dataset")
else:
    summary = (problems.groupby("dataset")["symbol"].agg(["count", lambda s: ", ".join(s[:6])])
               .rename(columns={"count": "symbols_flagged", "<lambda_0>": "examples"}))
    display(summary)
    display(problems.sort_values(["dataset", "symbol"]).head(25).reset_index(drop=True))

In [ ]:
# Symbol x month, one panel per dataset. A dark cell = events that month; a pale
# cell = none. Missing history reads as a solid blank block, so a half-ingested
# table is visible at a glance instead of hiding behind an average.
from plotly.subplots import make_subplots

months = pd.date_range(want_start.tz_localize(None), today.tz_localize(None), freq="MS")
order = sorted(uni)

fig = make_subplots(rows=1, cols=len(TABLES), shared_yaxes=True,
                    subplot_titles=list(TABLES), horizontal_spacing=0.012)
for i, label in enumerate(TABLES, start=1):
    frame = events[label]
    grid = pd.DataFrame(0.0, index=order, columns=months)
    if not frame.empty:
        counts = (frame.assign(m=frame["d"].dt.to_period("M").dt.to_timestamp())
                  .pivot_table(index="symbol", columns="m", values="n", aggfunc="sum"))
        grid = counts.reindex(index=order, columns=months).fillna(0.0)
    fig.add_trace(go.Heatmap(
        z=grid.to_numpy(), x=months, y=order,
        colorscale=[(0.0, "#f4f3ef"), (0.001, "#cfe0f5"), (1.0, SERIES[0])],
        zmin=0, zmax=max(float(grid.to_numpy().max()), 1.0), showscale=False,
        hovertemplate=(f"<b>{label}</b><br>%{{y}}<br>%{{x|%b %Y}}"
                       "<br>%{z:.0f} events<extra></extra>")), row=1, col=i)

fig.update_layout(
    title=(f"Alt-data coverage — {len(order)} names x {len(months)} months "
           f"(pale = no events; a blank BLOCK = missing history)"),
    template="plotly_white", height=max(420, 13 * len(order)),
    margin={"l": 70, "r": 20, "t": 90, "b": 40})
fig.update_yaxes(tickfont={"size": 8}, autorange="reversed")
fig.update_xaxes(tickfont={"size": 9})
fig.show()

## 6 · Schema sanity

Confirm the shared database still has each alt-data table with the columns the
feature layer (`trading_simple.features`) reads. These tables are defined in this
repo's `schema.sql` via `CREATE TABLE IF NOT EXISTS` — `insider_trades`,
`analyst_ratings` and `short_interest` are column-for-column identical to the
sibling repo's, and `analyst_price_targets` is **new here**, so this also catches
drift if the sibling altered anything.

In [ ]:
EXPECTED = {
    "insider_trades":        ["symbol", "txn_type", "shares", "price", "traded_at", "known_at"],
    "analyst_ratings":       ["symbol", "firm", "action", "grade_to", "rated_at", "known_at"],
    "analyst_price_targets": ["symbol", "firm", "pt_action", "pt_current", "pt_prior",
                              "rated_at", "known_at"],
    "short_interest":        ["symbol", "settlement_date", "days_to_cover", "known_at"],
}
with engine.connect() as c:
    for tbl, cols in EXPECTED.items():
        have = {r[0] for r in c.execute(text(
            "SELECT column_name FROM information_schema.columns WHERE table_name = :t"),
            {"t": tbl})}
        if not have:
            print(f"{tbl:22s} MISSING TABLE — run  uv run tsim init-db")
            continue
        missing = [col for col in cols if col not in have]
        print(f"{tbl:22s} {'OK' if not missing else 'missing ' + str(missing)}")

## Next steps

If §4 and §5 are green, open **`strategies/informed_flow/informed_flow.ipynb`** to
backtest the composite that ranks stocks on these signals — its §4 explains, name
by name, *why* a symbol entered or left the top bucket, and which of these
datasets drove it.

If anything was flagged, set `MODE = "backfill"` at the top and re-run — ingestion
is idempotent, so it only fills the holes.

The same work runs headless (but the audit above does not — prefer this notebook):

```bash
uv run tsim altdata --dataset all --universe LiquidLeaders --since 2020-01-01
```